# Phase 2 — Technical validation of the final models (13)

Issue #72. Notebook 15 (#71) scored every final model once on the official FD001 test set. This notebook asks which of the differences it found are real, and states the outcome plainly, including ties and null results. Nothing is trained or scored again: every number comes from the saved per-engine test predictions.

**Terms**

- **Engine error:** each of the 100 test engines is scored once, at its last recorded cycle, against its true remaining useful life (RUL). A model's RMSE is over those 100 errors.
- **Seeds:** the networks ran at seeds 42–46, the ML baselines once. With several seeds, RMSE is the mean of the per-seed RMSEs, and an engine's absolute error is averaged over seeds (the #56 toolkit, `rul.evaluation.validation`).
- **Bootstrap 95% CI:** resample the 100 engines with replacement 10,000 times and take the middle 95% of the resampled RMSEs. A *paired* bootstrap resamples the same engines for both models, so it measures the difference engine by engine.
- **Wilcoxon signed-rank test:** a test on the per-engine absolute errors of two models. **Holm's correction** adjusts the p-values for running several tests as one family.
- **Value test (#83, fixed in advance):** a hybrid adds value over its best single branch only if (1) the paired bootstrap CI on the RMSE difference lies below zero, or (2) the Holm-corrected Wilcoxon p is below 0.05 with the hybrid's median error lower, or (3) its RMSE varies less across seeds with a mean that is no worse. Otherwise the result is null.

**Which comparisons, chosen before looking at the test.** Every pairing below is fixed by cross-validation on the training engines, not by the test scores:

- **Best sequence model against best ML model:** the sequence model with the lowest cross-validated selection RMSE (#58–#60) against the ML model with the lowest (#61), both trained with the cap of 125.
- **Each hybrid against its best single branch:** the branch #68 found best in cross-validation (`fd001_hybrid_value_test.json`). The three comparisons form one Holm family.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from rul.evaluation.validation import bootstrap_ci, paired_bootstrap_diff, value_test

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
METRICS = ROOT / "results/metrics"
FIGURES = ROOT / "results/figures"
STAGES = {"late": (0, 50), "mid": (50, 100), "early": (100, np.inf)}  # true RUL, (low, high]

final = pd.read_csv(METRICS / "fd001_final_test_predictions.csv")
ml = pd.read_csv(METRICS / "fd001_ml_capped_test_predictions.csv").set_index("unit").sort_index()
units = ml.index.to_numpy()
y = ml["rul"].to_numpy(dtype=float)
assert (final.groupby("unit")["rul"].first().loc[units].to_numpy() == y).all(), "truth differs between files"

NETWORKS = {"lstm": "LSTM", "tcn": "Temporal CNN", "transformer": "Transformer",
            "hybrid_gated": "Hybrid A (gated)", "hybrid_two_stage": "Hybrid B (two-stage)"}
ML_NAMES = {"xgboost": "XGBoost", "random_forest": "Random Forest", "elastic_net": "Elastic Net"}
preds = {}  # (name, target) -> predictions shaped [seeds, engines]
for key, name in NETWORKS.items():
    for target in ("cap125", "uncapped"):
        wide = final.query("model == @key and target == @target").pivot(index="seed", columns="unit", values="prediction")
        preds[name, target] = wide[units].to_numpy()
for variant in ("engineered", "raw"):
    for key, name in ML_NAMES.items():
        for cap, target in (("125", "cap125"), ("none", "uncapped")):
            preds[f"{name} ({variant})", target] = ml[f"{variant}_{key}_cap{cap}"].to_numpy()[None, :]
print(f"{len(units)} test engines, true RUL {y.min():.0f} to {y.max():.0f}; "
      f"{len({n for n, _ in preds})} models, each capped and uncapped")

100 test engines, true RUL 7 to 145; 11 models, each capped and uncapped


## 1. Every model: seed spread and bootstrap CIs

`RMSE ± s.d.` is the mean and standard deviation over seeds (networks only). The CI is the bootstrap 95% interval over engines for the seed-averaged RMSE, so it shows how much the score depends on which 100 engines happen to be in the test set.

In [2]:
def seed_rmse(p):
    return np.sqrt(np.mean((p - y) ** 2, axis=1))


rows = []
for (name, target), p in preds.items():
    r = seed_rmse(p)
    ci = bootstrap_ci(y, p, "rmse")
    mae = bootstrap_ci(y, p, "mae")
    rows.append({"model": name, "target": target, "seeds": len(r),
                 "RMSE mean": r.mean(), "RMSE s.d.": r.std(ddof=1) if len(r) > 1 else np.nan,
                 "RMSE CI low": ci["low"], "RMSE CI high": ci["high"],
                 "MAE": mae["estimate"], "MAE CI low": mae["low"], "MAE CI high": mae["high"]})
models = pd.DataFrame(rows)
models.round(3).to_csv(METRICS / "fd001_validation_models.csv", index=False)


def show(target):
    t = models[models.target == target].sort_values("RMSE mean").set_index("model")
    return pd.DataFrame({
        "seeds": t["seeds"],
        "RMSE ± s.d.": [f"{m:.2f}" + (f" ± {s:.2f}" if pd.notna(s) else "") for m, s in zip(t["RMSE mean"], t["RMSE s.d."])],
        "RMSE 95% CI": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(t["RMSE CI low"], t["RMSE CI high"])],
        "MAE 95% CI": [f"{m:.2f} [{lo:.2f}, {hi:.2f}]" for m, lo, hi in zip(t["MAE"], t["MAE CI low"], t["MAE CI high"])],
    })


print("Trained with the cap of 125")
display(show("cap125"))
print("Trained uncapped")
display(show("uncapped"))

Trained with the cap of 125


,seeds,RMSE ± s.d.,RMSE 95% CI,MAE 95% CI
model,,,,
Hybrid B (two-stage),5,15.18 ± 0.55,"[12.79, 17.47]","11.28 [9.38, 13.28]"
Hybrid A (gated),5,15.40 ± 0.46,"[13.15, 17.73]","11.89 [10.07, 13.79]"
Transformer,5,15.60 ± 0.35,"[13.38, 17.72]","11.91 [10.08, 13.82]"
XGBoost (engineered),1,15.71,"[13.24, 18.05]","11.59 [9.55, 13.73]"
Temporal CNN,5,15.83 ± 0.55,"[13.73, 17.88]","12.10 [10.35, 13.99]"
LSTM,5,15.95 ± 0.33,"[13.49, 18.36]","11.96 [10.01, 14.02]"
Random Forest (engineered),1,16.21,"[13.64, 18.64]","11.95 [9.86, 14.14]"
XGBoost (raw),1,17.58,"[14.70, 20.20]","12.78 [10.45, 15.18]"
Elastic Net (engineered),1,17.80,"[15.83, 19.62]","14.83 [12.90, 16.74]"


Trained uncapped


,seeds,RMSE ± s.d.,RMSE 95% CI,MAE 95% CI
model,,,,
XGBoost (raw),1,24.83,"[20.73, 28.79]","18.09 [14.84, 21.50]"
Hybrid B (two-stage),5,25.86 ± 1.07,"[20.93, 30.47]","17.71 [14.30, 21.36]"
Random Forest (raw),1,26.15,"[21.83, 30.36]","19.22 [15.87, 22.68]"
Hybrid A (gated),5,27.52 ± 1.10,"[22.14, 33.17]","19.75 [16.31, 23.60]"
Random Forest (engineered),1,27.55,"[21.07, 34.15]","17.67 [13.77, 22.07]"
Elastic Net (engineered),1,28.39,"[24.82, 32.08]","23.57 [20.56, 26.75]"
XGBoost (engineered),1,28.56,"[22.58, 34.46]","18.96 [14.95, 23.41]"
LSTM,5,28.97 ± 4.98,"[23.31, 34.41]","19.43 [15.69, 23.47]"
Elastic Net (raw),1,29.64,"[26.00, 33.15]","24.44 [21.19, 27.74]"
